# Desarrollo de Aplicaciones Web (DAB-2201)
## Sesión 19: Sanitización de Entradas, Prevención de XSS e Inyecciones
**Docente Titular:** MCC. Iván Márquez Larios (`ijmarquezl`)
**Repositorio Maestro:** `https://github.com/ijmarquezl/tecnm-desarrollo-web`

---
### Protocolo de la Sesión
1. **Fase 1 (El Artesano):** Resolver a mano: *Limpieza manual de HTML y caracteres especiales*.
2. **Fase 2 (El Científico):** Definir contrato C-R-E-O: *Sanitizador de entradas y escapes de salida JSON/HTML* y formular prompt.
3. **Fase 3 (El Auditor):** Someter el servidor al Test Harness: *Pruebas de inyección de scripts en payloads*.
4. **Entrega:** Documentar en `auditorias/auditoria_sesion19.md`.

### Paso 0: Inicialización del Entorno (FastAPI y PHP 8+)

In [1]:
import os
print('⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...')
!pip install -q fastapi uvicorn httpx pydantic passlib python-jose[cryptography] sqlalchemy alembic
if os.system('which php > /dev/null 2>&1') != 0:
    print('⏳ Instalando PHP CLI...')
    !apt-get update -qq && apt-get install -y -qq php-cli > /dev/null 2>&1
print('✅ Entorno Web listo:')
!python3 --version
!php -v | head -n 1

⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...
⏳ Instalando PHP CLI...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ Entorno Web listo:
Python 3.13.18
PHP 8.3.6 (cli) (built: Nov 18 2026 23:42:07) (NTS)


---
## SECCIÓN A: TRACK PYTHON / FASTAPI
### Fase 1: El Artesano (15 min - Sin IA)
**Reto:** Limpieza manual de HTML y caracteres especiales

In [2]:
%%writefile app_artesano.py
import re
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel

app = FastAPI(title="API Artesano")


class PayloadRequest(BaseModel):
    dato: str


def limpiar_entrada(texto: str) -> str:
    limpio = re.sub(r"<[^>]+>", "", texto)
    limpio = limpio.replace("&", "&amp;")
    limpio = limpio.replace("<", "&lt;")
    limpio = limpio.replace(">", "&gt;")
    limpio = limpio.replace('"', "&quot;")
    limpio = limpio.replace("'", "&#39;")
    return limpio.strip()


@app.post("/api/recurso")
def procesar_recurso(payload: PayloadRequest):
    if not payload.dato.strip():
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Dato vacio"
        )

    texto_limpio = limpiar_entrada(payload.dato)
    if not texto_limpio:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Dato vacio"
        )

    return {"status": "success", "mensaje": f"Procesado: {texto_limpio}"}


Writing app_artesano.py


### Fase 2: El Científico (Prompting C-R-E-O)
Pega aquí la implementación optimizada propuesta por la IA:

In [3]:
%%writefile app_ia.py
import html
import re
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel

app = FastAPI(title="API Cientifico")


class PayloadRequest(BaseModel):
    dato: str


def limpiar_y_escapar(texto: str) -> str:
    sin_script = re.sub(r"(?is)<(script|style)[^>]*>.*?</\1>", "", texto)
    sin_tags = re.sub(r"<[^>]+>", "", sin_script)
    sin_esquemas = re.sub(r"(?i)(javascript|data)\s*:", "", sin_tags)
    return html.escape(sin_esquemas.strip(), quote=True)


@app.post("/api/recurso")
def procesar_recurso(payload: PayloadRequest):
    if not payload.dato.strip():
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Dato vacio",
        )

    limpio = limpiar_y_escapar(payload.dato)
    if not limpio:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Dato vacio",
        )

    return {"status": "success", "mensaje": f"Procesado: {limpio}"}

Writing app_ia.py


### Fase 3: El Auditor (Test Harness HTTP)
Pruebas de servidor: Pruebas de inyección de scripts en payloads

In [4]:
import threading
import time
import httpx
import uvicorn

from app_artesano import app as app_artesano
from app_ia import app as app_ia


def levantar(app_obj, puerto):
    cfg = uvicorn.Config(app_obj, host="127.0.0.1", port=puerto, log_level="error")
    srv = uvicorn.Server(cfg)
    threading.Thread(target=srv.run, daemon=True).start()
    time.sleep(1)


def probar_inyecciones(nombre, url):
    payloads = [
        "hola normal",
        "<script>alert('xss')</script>",
        '<img src="x" onerror="alert(1)">',
        "<svg onload=alert(1)>texto</svg>",
        "javascript:alert(1)",
    ]

    print(f"=== {nombre} ({url}) ===")
    with httpx.Client(base_url=url, timeout=5.0) as cliente:
        for p in payloads:
            r = cliente.post("/api/recurso", json={"dato": p})
            print(f"Entrada: {p!r} -> HTTP {r.status_code} | {r.json()}")


levantar(app_artesano, 8021)
levantar(app_ia, 8022)

probar_inyecciones("app_artesano", "http://127.0.0.1:8021")
probar_inyecciones("app_ia", "http://127.0.0.1:8022")


=== app_artesano (http://127.0.0.1:8021) ===
Entrada: 'hola normal' -> HTTP 200 | {'status': 'success', 'mensaje': 'Procesado: hola normal'}
Entrada: "<script>alert('xss')</script>" -> HTTP 200 | {'status': 'success', 'mensaje': 'Procesado: alert(&#39;xss&#39;)'}
Entrada: '<img src="x" onerror="alert(1)">' -> HTTP 400 | {'detail': 'Dato vacio'}
Entrada: '<svg onload=alert(1)>texto</svg>' -> HTTP 200 | {'status': 'success', 'mensaje': 'Procesado: texto'}
Entrada: 'javascript:alert(1)' -> HTTP 200 | {'status': 'success', 'mensaje': 'Procesado: javascript:alert(1)'}
=== app_ia (http://127.0.0.1:8022) ===
Entrada: 'hola normal' -> HTTP 200 | {'status': 'success', 'mensaje': 'Procesado: hola normal'}
Entrada: "<script>alert('xss')</script>" -> HTTP 400 | {'detail': 'Dato vacio'}
Entrada: '<img src="x" onerror="alert(1)">' -> HTTP 400 | {'detail': 'Dato vacio'}
Entrada: '<svg onload=alert(1)>texto</svg>' -> HTTP 200 | {'status': 'success', 'mensaje': 'Procesado: texto'}
Entrada: 'javascript: